# 🌾 MicroSegment — Rural Micro-Entrepreneur Segmentation & Advisory Engine
### INT 396 Capstone Project Walkthrough
**Author:** Senior Data Science & Rural Finance Analytics Consultant

---
## Executive Context & Objectives
Rural micro-entrepreneurs accessing concessional credit (MUDRA, NRLM, SHG-Bank linkage, RRBs) are currently treated as one **monolithic cohort**. This project:
1. Discovers structurally meaningful segments using **unsupervised clustering** regularized with **Yeo-Johnson power transformations**.
2. Empirically validates the optimal cluster count ($k=5$) via Silhouette, Davies-Bouldin, and $B=100$ bootstrap stability analysis.
3. Translates segments into **operational persona dossiers** and tailored credit blueprints.
4. Deploys a **real-time production classifier** (5-Fold CV Macro F1: $98.8\%$) and interactive **Streamlit web application**.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

print('Libraries successfully imported!')

## 1. Data Inspection & Statistical Calibration
The dataset comprises $N=4,000$ rural micro-entrepreneurs calibrated against empirical NABARD/MUDRA survey benchmarks.

In [ ]:
df_raw = pd.read_csv('../data/raw/rural_micro_entrepreneurs_synthetic.csv')
print(f'Raw Dataset Shape: {df_raw.shape}')
df_raw[['entrepreneur_id', 'business_sector', 'monthly_revenue', 'loan_amount', 'repayment_on_time_ratio', 'shg_cooperative_member']].head()

## 2. Feature Engineering & Yeo-Johnson Regularization
Raw financial distributions exhibit severe positive skewness (`monthly_revenue` skew = $+2.78$). Applying Yeo-Johnson regularizes skewness to $0.006$, enabling distortion-free Euclidean clustering.

In [ ]:
df_eng = pd.read_csv('../data/processed/rural_micro_entrepreneurs_engineered.csv')
df_scaled = pd.read_csv('../data/processed/rural_micro_entrepreneurs_scaled_features.csv')
print(f'Engineered Features Matrix Shape: {df_scaled.shape}')
df_eng[['repayment_ratio', 'revenue_to_loan_ratio', 'income_stability', 'business_maturity', 'financial_inclusion_score', 'digital_adoption_score']].describe().T

## 3. Multi-Metric Unsupervised Model Selection ($k=2 \dots 10$)
Benchmarking K-Means, Gaussian Mixture Models, and Ward Agglomerative clustering across multiple mathematical criteria.

In [ ]:
metrics_df = pd.read_csv('../models/clustering_metrics.csv')
km_metrics = metrics_df[metrics_df['algorithm'] == 'KMeans'].sort_values('k')
print('K-Means Metric Sweep Across k=2..10:')
km_metrics[['k', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'inertia']]

### Key Validation Insight:
At $k=5$, Silhouette score reaches its global maximum ($0.2515$) and Davies-Bouldin reaches its global minimum ($1.4442$).

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(km_metrics['k'], km_metrics['silhouette'], 'b-o', label='Silhouette (Higher is Better)', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Silhouette Score', color='b')
ax1.axvline(5, color='r', linestyle='--', label='Selected k=5')
ax2 = ax1.twinx()
ax2.plot(km_metrics['k'], km_metrics['davies_bouldin'], 'g-s', label='Davies-Bouldin (Lower is Better)', linewidth=2)
ax2.set_ylabel('Davies-Bouldin Index', color='g')
plt.title('Cluster Validation: Silhouette and Davies-Bouldin vs. k', weight='bold')
plt.show()

## 4. Persona Dossiers & Statistical Profiling
Examining the 5 discovered segments across financial scale, credit absorption, repayment integrity, and digital adoption.

In [ ]:
summary_df = pd.read_csv('../reports/segment_statistical_summary.csv')
summary_df

## 5. Real-Time Production Classifier & Advisory Scoring
Testing the single-sample inference pipeline on a new applicant profile.

In [ ]:
from src.classifier import RealTimeSegmentInferenceEngine
engine = RealTimeSegmentInferenceEngine('../models/production_classifier.joblib')

sample_applicant = {
    'age': 26, 'gender': 'Male', 'education': 'Graduate', 'location_type': 'Semi-Urban', 'region': 'South',
    'business_sector': 'Services & Repair', 'years_in_business': 2.0, 'loan_amount': 150000.0,
    'tenure_months': 24, 'interest_concession_pct': 2.5, 'monthly_revenue': 48000.0,
    'income_variability': 0.40, 'num_employees': 1, 'repayment_on_time_ratio': 0.88,
    'num_delayed_installments': 1, 'has_defaulted_past': 0, 'mobile_banking_usage': 1,
    'digital_payment_ratio': 0.85, 'savings_frequency': 'Weekly', 'savings_balance_ratio': 0.20,
    'shg_cooperative_member': 0, 'market_access_type': 'Direct Consumer', 'past_training_received': 1
}

prediction = engine.predict(sample_applicant)
print(f"Predicted Segment: {prediction['predicted_segment_id']} — {prediction['archetype_name']}")
print(f"Model Confidence: {prediction['confidence']*100:.2f}%")
print(f"Risk Tier: {prediction['advisory_card']['risk_assessment']['risk_tier']}")
print(f"Recommended Product: {prediction['advisory_card']['credit_blueprint']['product_name']}")
print(f"Top Action: {prediction['advisory_card']['action_items'][0]['action']}")

## 6. Conclusion & Viva Takeaways
1. **Monolithic credit treatment fails** in priority sector lending: high-growth scalers are starved of growth capital, while vulnerable subsistence producers are forced into rigid amortizations.
2. **MicroSegment delivers targeted capacity building** before default occurs.
3. **Production ready:** 5-Fold Stratified CV achieves $98.8\%$ Macro F1 for real-time single applicant scoring.